In [ ]:
!pip install -q ultralytics mlflow scikit-learn pyyaml roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.6/333.6 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import os

!git clone https://github.com/Yomnahaouel/signlive-asl-detection.git
os.chdir("/content/signlive-asl-detection")

from roboflow import Roboflow
rf = Roboflow(api_key="Mj6DkUy3Jorc2WevjwcC")
project = rf.workspace("david-lee-d0rhs").project("american-sign-language-letters")
version = project.version(1)
dataset = version.download("yolov8", location="data/raw/asl_letters", overwrite=True)

# Vérification
from pathlib import Path
raw = Path("data/raw/asl_letters")
for split in ["train", "valid", "test"]:
    img_dir = raw / split / "images"
    count = len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0
    print(f"{split}/images : {count} fichiers")

Cloning into 'signlive-asl-detection'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 91 (delta 15), reused 82 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 131.30 KiB | 1.06 MiB/s, done.
Resolving deltas: 100% (15/15), done.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/asl_letters in yolov8:: 100%|██████████| 3468/3468 [00:00<00:00, 12126.44it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
train/images : 1512 fichiers
valid/images : 144 fichiers
test/images : 72 fichiers


In [ ]:
import shutil, hashlib
from pathlib import Path
from sklearn.model_selection import train_test_split
import yaml

RAW_DIR = Path("data/raw/asl_letters")
PROCESSED_DIR = Path("data/processed")
SEED = 42
CLASSES = [chr(i) for i in range(ord("A"), ord("Z") + 1)]

def get_image_hash(p):
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def collect_samples(raw_dir):
    samples = []
    for split in ["train", "valid", "test"]:
        img_dir = raw_dir / split / "images"
        lbl_dir = raw_dir / split / "labels"
        if not img_dir.exists():
            continue
        for img_path in sorted(img_dir.glob("*.jpg")):
            lbl_path = lbl_dir / img_path.with_suffix(".txt").name
            if lbl_path.exists():
                samples.append({"image": img_path, "label": lbl_path})
    print(f"✅ {len(samples)} paires collectées")
    return samples

def deduplicate(samples):
    seen, unique = set(), []
    for s in samples:
        h = get_image_hash(s["image"])
        if h not in seen:
            seen.add(h)
            unique.append(s)
    print(f"✅ {len(unique)} uniques")
    return unique

def copy_samples(samples, dest_dir, split_name):
    img_dir = dest_dir / split_name / "images"
    lbl_dir = dest_dir / split_name / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for s in samples:
        shutil.copy(s["image"], img_dir / s["image"].name)
        shutil.copy(s["label"], lbl_dir / s["label"].name)
    print(f"✅ {len(samples)} copiés dans {split_name}/")

samples = collect_samples(RAW_DIR)
samples = deduplicate(samples)
train_val, test = train_test_split(samples, test_size=0.15, random_state=SEED)
train, valid = train_test_split(train_val, test_size=0.176, random_state=SEED)
print(f"📊 train={len(train)}, valid={len(valid)}, test={len(test)}")

if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
PROCESSED_DIR.mkdir(parents=True)

copy_samples(train, PROCESSED_DIR, "train")
copy_samples(valid, PROCESSED_DIR, "valid")
copy_samples(test, PROCESSED_DIR, "test")

config = {
    "train": str((PROCESSED_DIR / "train" / "images").resolve()),
    "val":   str((PROCESSED_DIR / "valid" / "images").resolve()),
    "test":  str((PROCESSED_DIR / "test"  / "images").resolve()),
    "nc": 26,
    "names": CLASSES,
}
with open(PROCESSED_DIR / "data.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print("✅ data.yaml généré !")

✅ 1728 paires collectées
✅ 1728 uniques
📊 train=1209, valid=259, test=260
✅ 1209 copiés dans train/
✅ 259 copiés dans valid/
✅ 260 copiés dans test/
✅ data.yaml généré !


In [ ]:
import os
import mlflow

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/haouelyomna111/signlive-asl-detection.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "haouelyomna111"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "58d18129d0e53d88fa67b8edc209645fe2e1c420"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("signlive")
print("✅ MLflow configuré !")

# Vérification GPU
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

✅ MLflow configuré !
GPU disponible : True
GPU : Tesla T4


In [ ]:
from ultralytics import YOLO
import mlflow, os, torch

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("signlive")

with mlflow.start_run(run_name="yolo11n-baseline-2"):
    model = YOLO("yolo11n.pt")

    results = model.train(
        data="data/processed/data.yaml",
        epochs=50,
        imgsz=640,
        batch=16,
        seed=42,
        device=0 if torch.cuda.is_available() else "cpu",
        name="yolo11n-baseline-2",
        project="runs/train",
    )

    # Logger seulement les métriques finales
    metrics = results.results_dict
    mlflow.log_metrics({
        "mAP50":     metrics["metrics/mAP50(B)"],
        "mAP50_95":  metrics["metrics/mAP50-95(B)"],
        "precision": metrics["metrics/precision(B)"],
        "recall":    metrics["metrics/recall(B)"],
    })

    print("✅ Entraînement terminé !")
    print(f"mAP50    : {metrics['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95 : {metrics['metrics/mAP50-95(B)']:.4f}")

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n-baseline-2-2, nbs=64, nms=None, opset=None, optimize=Fal

2026/10/01 20:23:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/10/01 20:23:35 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/10/01 20:23:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(1de883cdf5894767988ac177243ac56b) to https://dagshub.com/haouelyomna111/signlive-asl-detection.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Using 1209 train, 259 val images for fraction=1.0 at imgsz=640
Using 2 dataloader workers
Logging results to /content/signlive-asl-detection/runs/detect/runs/train/yolo11n-baseline-2-2
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      2.55G      1.008      4.339      1.454         22        640: 100% ━━━━━━━━━━━━ 76/76 3.9it/s 19.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.6it/s 2.5s
                   all        259        259     0.0351      0.611     0.0584     0.0483

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.98G     0.8466      3.971      1.288         22        640: 100% ━━━━━━━━━━━━ 76/76 4.7it/s 16.3s

In [ ]:
from ultralytics import YOLO
import mlflow, os, torch

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("signlive")

with mlflow.start_run(run_name="yolo26n-advanced"):
    model = YOLO("yolo26n.pt")  # YOLO26 pré-entraîné

    results = model.train(
        data="data/processed/data.yaml",
        epochs=50,
        imgsz=640,
        batch=16,
        seed=42,
        device=0 if torch.cuda.is_available() else "cpu",
        name="yolo26n-advanced",
        project="runs/train",
    )

    metrics = results.results_dict
    mlflow.log_metrics({
        "mAP50":     metrics["metrics/mAP50(B)"],
        "mAP50_95":  metrics["metrics/mAP50-95(B)"],
        "precision": metrics["metrics/precision(B)"],
        "recall":    metrics["metrics/recall(B)"],
    })

    print("✅ YOLO26n terminé !")
    print(f"mAP50    : {metrics['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95 : {metrics['metrics/mAP50-95(B)']:.4f}")
    print(f"Précision: {metrics['metrics/precision(B)']:.4f}")
    print(f"Rappel   : {metrics['metrics/recall(B)']:.4f}")

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n-advanced, nbs=64, nms=None, opset=None, optimize=False, 

2026/10/01 20:43:07 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/10/01 20:43:07 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/10/01 20:43:07 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(c9214350f9a248278f6df5e25b692eaa) to https://dagshub.com/haouelyomna111/signlive-asl-detection.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Using 1209 train, 259 val images for fraction=1.0 at imgsz=640
Using 2 dataloader workers
Logging results to /content/signlive-asl-detection/runs/detect/runs/train/yolo26n-advanced
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       1/50      2.69G     0.9392      7.768    0.01819         22        640: 100% ━━━━━━━━━━━━ 76/76 3.5it/s 21.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.5it/s 2.6s
                   all        259        259     0.0494      0.193     0.0362     0.0286

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       2/50      3.12G     0.7686      7.184    0.01461         22        640: 100% ━━━━━━━━━━━━ 76/76 4.0it/s 19.1s
   

In [ ]:
import os
from pathlib import Path

runs = Path("/content/signlive-asl-detection/runs")
for pt in runs.rglob("best.pt"):
    print(pt)